# Pairs Trading — Notebook 2: Backtest (Single-Position, H1 + H4)

هدف: ارزیابی edge هر کاندیدا از نوت‌بوک ۲۵ به‌صورت تک‌پوزیشن (پورتفولیو-سازی در نوت‌بوک ۲۷).

## استراتژی
برای هر spread cointegrated:
- **Spread:** `s_t = log(P_y_t) - β · log(P_x_t) - α`  (β و α از in-sample در نوت‌بوک ۲۵ ثابت)
- **Z-score رولینگ:** `z_t = (s_t - μ_t) / σ_t` با `μ_t, σ_t` فقط از past data — **بدون lookahead**
- **سیگنال (state-machine):**
    - `z > +entry_z` → SHORT spread (short y، long x به نسبت β)
    - `z < -entry_z` → LONG spread (long y، short x)
    - `|z| < exit_z` → خروج (take profit / converge)
    - `|z| > stop_z` → خروج (loss / structural break)
    - time-stop = 4 × half-life — اگه بیشتر طول کشید، خروج اجباری

## هزینه‌ها (per-leg، per-round-turn)
- **Spread cost** (پیپ entry+exit): تقریبی، از ستون `spread` در CSV یا constant
- **Commission:** ثابت per round-turn
- **Swap:** هزینه‌ی نگه‌داری روزانه (مهم چون positions چندهفته‌ای)

## متریک‌ها (per spread)
- total return، Sharpe (annualized)، max drawdown، hit rate
- avg holding period، n trades، PnL gross vs net
- در دو پنجره: in-sample (2023-2025) و out-of-sample (2026-)

**ورودی:** `notebooks/data/stat_arb/cointegrated_shortlist_{H1,H4}.csv`
**خروجی:** `notebooks/data/stat_arb/backtest_results_{H1,H4}.csv` + `backtest_trades_{H1,H4}.csv`

In [ ]:
from __future__ import annotations
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("ready")

ready


## Configuration

In [ ]:
TIMEFRAMES           = ["H1", "H4"]
IN_SAMPLE_START      = "2023-01-01"
IN_SAMPLE_END        = "2025-12-31"
OUT_OF_SAMPLE_START  = "2026-01-01"

# Strategy thresholds (z-score in standard deviations of the rolling spread)
ENTRY_Z              = 2.0
EXIT_Z               = 0.5
STOP_Z               = 4.0
Z_WINDOW_MULT        = 4          # rolling z-window = 4 × half_life_bars
Z_WINDOW_MIN         = 100        # but never below this
TIME_STOP_MULT       = 4          # time-stop = 4 × half_life_bars

# Cost assumptions (per leg, in basis points of notional)
# Realistic for retail MT5: spread cost ≈ 1 pip on majors / leg → ~1 bp on a 10000-ish quote.
# Commission ≈ $7 / lot round-turn ≈ ~0.5 bp on $100k notional.
# Swap is highly broker-dependent; assume small negative carry.
SPREAD_COST_BPS_PER_LEG_PER_SIDE = 1.0   # entry+exit doubled inside the engine
COMMISSION_BPS_PER_LEG_PER_SIDE  = 0.5
SWAP_BPS_PER_LEG_PER_DAY         = 0.5   # negative carry on both legs (typical worst case)

DATA_DIR   = PROJECT_ROOT / "notebooks" / "data"
STAT_DIR   = DATA_DIR / "stat_arb"
REAL_TZ    = "Europe/Nicosia"
TF_HOURS   = {"H1": 1, "H4": 4, "D1": 24}
BARS_PER_YEAR = {"H1": 24*252, "H4": 6*252, "D1": 252}   # FX has ~252 trading days

print(f"thresholds:  entry=±{ENTRY_Z}σ  exit=±{EXIT_Z}σ  stop=±{STOP_Z}σ")
print(f"window:      {Z_WINDOW_MULT}×half_life (min {Z_WINDOW_MIN} bars)")
print(f"time-stop:   {TIME_STOP_MULT}×half_life")
print(f"costs/leg:   spread={SPREAD_COST_BPS_PER_LEG_PER_SIDE}bps  comm={COMMISSION_BPS_PER_LEG_PER_SIDE}bps  swap={SWAP_BPS_PER_LEG_PER_DAY}bps/day")

thresholds:  entry=±2.0σ  exit=±0.5σ  stop=±4.0σ
window:      4×half_life (min 100 bars)
time-stop:   4×half_life
costs/leg:   spread=1.0bps  comm=0.5bps  swap=0.5bps/day


## ۱) Loader (re-use logic from notebook 25)

In [ ]:
def load_pair_h1(symbol: str) -> pd.Series:
    path = DATA_DIR / symbol / "H1" / "ohlcv.csv"
    df = pd.read_csv(path, parse_dates=["time"])
    naive = df["time"].dt.tz_localize(None)
    ts = naive.dt.tz_localize(REAL_TZ, ambiguous="NaT", nonexistent="NaT")
    s = pd.Series(df["close"].values, index=ts, name=symbol)
    return s[s.index.notna()].sort_index()


def to_tf(prices_h1: pd.DataFrame, tf: str) -> pd.DataFrame:
    if tf == "H1":
        return prices_h1
    rule = {"H4": "4h", "D1": "1D"}[tf]
    return prices_h1.resample(rule, label="right", closed="right").last().dropna()


# Load every symbol that appears in any shortlist (we only need those).
shortlists = {tf: pd.read_csv(STAT_DIR / f"cointegrated_shortlist_{tf}.csv") for tf in TIMEFRAMES}
needed = set()
for sl in shortlists.values():
    needed |= set(sl["y"]).union(sl["x"])
needed = sorted(needed)
print(f"shortlist sizes: {{tf: len for tf, len in {{k: len(v) for k, v in shortlists.items()}}.items()}}")
print(f"symbols needed:  {len(needed)} → {needed}")

h1 = {sym: load_pair_h1(sym) for sym in needed}
all_h1 = pd.concat(h1, axis=1, sort=True).dropna()
print(f"H1 aligned full history: {all_h1.shape}")

prices_by_tf = {tf: to_tf(all_h1, tf) for tf in TIMEFRAMES}
for tf, p in prices_by_tf.items():
    print(f"  [{tf}] bars={len(p)}  range={p.index.min()} → {p.index.max()}")

shortlist sizes: {tf: len for tf, len in {k: len(v) for k, v in shortlists.items()}.items()}
symbols needed:  28 → ['AUDCAD', 'AUDCHF', 'AUDJPY', 'AUDNZD', 'AUDUSD', 'CADCHF', 'CADJPY', 'CHFJPY', 'EURAUD', 'EURCAD', 'EURCHF', 'EURGBP', 'EURJPY', 'EURNZD', 'EURUSD', 'GBPAUD', 'GBPCAD', 'GBPCHF', 'GBPJPY', 'GBPNZD', 'GBPUSD', 'NZDCAD', 'NZDCHF', 'NZDJPY', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY']
H1 aligned full history: (24877, 28)
  [H1] bars=24877  range=2022-05-16 23:00:00+03:00 → 2026-05-15 23:00:00+03:00
  [H4] bars=6356  range=2022-05-17 00:00:00+03:00 → 2026-05-16 00:00:00+03:00


## ۲) Backtest engine — state machine on z-score

Vectorized rolling stats، loop ساده روی bars برای state transitions. PnL در log-space (≈ percentage return for small spreads).

**نکات مهم:**
- rolling mean/std فقط روی past data → بدون lookahead
- ورود و خروج روی close همان بار (سادگی؛ شیفت یک‌بار به جلو برای واقع‌گرایی در نسخه‌ی بعد)
- duration در بار → روز برای محاسبه‌ی swap
- PnL gross در log-space سپس کسر هزینه‌ها در bps

In [ ]:
@dataclass
class Trade:
    entry_time: pd.Timestamp
    exit_time:  pd.Timestamp
    side:       int            # +1 long spread, -1 short spread
    entry_spread: float
    exit_spread:  float
    entry_z:    float
    exit_z:     float
    gross_pnl:  float          # log-space, ≈ fraction return
    cost:       float          # bps converted to fraction
    net_pnl:    float
    duration_bars: int
    duration_days: float
    exit_reason:  str          # 'mean_revert', 'stop_z', 'time_stop'


def backtest_one_spread(
    y_sym: str, x_sym: str, alpha: float, beta: float,
    half_life_bars: float, prices: pd.DataFrame, tf: str,
) -> tuple[list[Trade], pd.Series]:
    """Single-spread backtest. Returns (trades, equity_curve_bps)."""
    bar_hours = TF_HOURS[tf]
    z_window  = max(int(round(half_life_bars * Z_WINDOW_MULT)), Z_WINDOW_MIN)
    time_stop = max(int(round(half_life_bars * TIME_STOP_MULT)), Z_WINDOW_MIN)

    py, px = prices[y_sym], prices[x_sym]
    spread = np.log(py) - beta * np.log(px) - alpha

    # rolling stats — strictly past-looking (closed='left' would shift, but rolling already excludes the current bar's future)
    mu = spread.rolling(z_window, min_periods=z_window).mean()
    sd = spread.rolling(z_window, min_periods=z_window).std()
    z  = (spread - mu) / sd
    valid = z.dropna()
    if valid.empty:
        return [], pd.Series(dtype=float)

    # Cost per round-trip in fraction units (bps → /10000)
    # Two legs, two sides each (entry + exit). Spread + commission.
    legs_per_rt   = 2 * 2     # two legs × (entry + exit)
    fixed_cost_bp = legs_per_rt * (SPREAD_COST_BPS_PER_LEG_PER_SIDE + COMMISSION_BPS_PER_LEG_PER_SIDE)
    fixed_cost    = fixed_cost_bp / 1e4

    trades: list[Trade] = []
    position = 0                            # +1 long spread / -1 short / 0 flat
    entry_t = None
    entry_spread = entry_z_val = None
    entry_idx = -1

    times = valid.index.to_numpy()
    z_vals = valid.to_numpy()
    s_vals = spread.reindex(valid.index).to_numpy()

    def close_trade(i: int, reason: str) -> None:
        nonlocal position, entry_t, entry_spread, entry_z_val, entry_idx
        exit_t = pd.Timestamp(times[i])
        exit_s = float(s_vals[i])
        exit_z_val = float(z_vals[i])
        # gross PnL in log-space: long spread profits if spread rises
        gross = position * (exit_s - entry_spread)
        duration_bars = i - entry_idx
        duration_days = duration_bars * bar_hours / 24.0
        swap_cost = 2 * SWAP_BPS_PER_LEG_PER_DAY * duration_days / 1e4
        total_cost = fixed_cost + swap_cost
        net = gross - total_cost
        trades.append(Trade(
            entry_time=entry_t, exit_time=exit_t, side=position,
            entry_spread=entry_spread, exit_spread=exit_s,
            entry_z=entry_z_val, exit_z=exit_z_val,
            gross_pnl=gross, cost=total_cost, net_pnl=net,
            duration_bars=duration_bars, duration_days=duration_days,
            exit_reason=reason,
        ))
        position = 0
        entry_t = entry_spread = entry_z_val = None
        entry_idx = -1

    for i, z_t in enumerate(z_vals):
        if position == 0:
            if z_t > ENTRY_Z:
                position, entry_idx = -1, i
                entry_t = pd.Timestamp(times[i])
                entry_spread, entry_z_val = float(s_vals[i]), float(z_t)
            elif z_t < -ENTRY_Z:
                position, entry_idx = +1, i
                entry_t = pd.Timestamp(times[i])
                entry_spread, entry_z_val = float(s_vals[i]), float(z_t)
        else:
            duration_bars = i - entry_idx
            if abs(z_t) >= STOP_Z and ((position == 1 and z_t < -ENTRY_Z) or (position == -1 and z_t > ENTRY_Z)):
                # adverse move past STOP_Z in same direction → stop out
                close_trade(i, "stop_z")
            elif duration_bars >= time_stop:
                close_trade(i, "time_stop")
            elif abs(z_t) <= EXIT_Z:
                close_trade(i, "mean_revert")

    # If still in position at end, force exit at last bar (no cost-of-not-exiting bias)
    if position != 0:
        close_trade(len(z_vals) - 1, "end_of_data")

    # Equity curve: cumulative net PnL marked at exit time
    if trades:
        equity = pd.Series({t.exit_time: t.net_pnl for t in trades}).sort_index().cumsum()
    else:
        equity = pd.Series(dtype=float)
    return trades, equity


print("backtest engine defined.")

backtest engine defined.


## ۳) Metric helpers

In [ ]:
def trades_to_df(trades: list[Trade]) -> pd.DataFrame:
    if not trades:
        return pd.DataFrame(columns=["entry_time", "exit_time", "side", "net_pnl",
                                     "gross_pnl", "cost", "duration_days", "exit_reason"])
    return pd.DataFrame([t.__dict__ for t in trades])


def split_by_sample(df: pd.DataFrame, time_col: str = "exit_time") -> tuple[pd.DataFrame, pd.DataFrame]:
    in_mask = (df[time_col] >= IN_SAMPLE_START) & (df[time_col] <= IN_SAMPLE_END)
    oos_mask = df[time_col] >= OUT_OF_SAMPLE_START
    return df[in_mask].reset_index(drop=True), df[oos_mask].reset_index(drop=True)


def metrics(tdf: pd.DataFrame, tf: str, label: str = "") -> dict:
    """Per-spread metrics on a trade-set."""
    if tdf.empty:
        return dict(label=label, n_trades=0, total_pnl=0.0, total_cost=0.0,
                    avg_pnl=0.0, hit_rate=0.0, avg_duration_days=0.0,
                    sharpe=0.0, max_dd=0.0)
    pnl = tdf["net_pnl"]
    eq = pnl.cumsum()
    running_max = eq.cummax()
    dd = (eq - running_max).min()
    # Sharpe per trade then annualize — use trades/year derived from avg duration
    if pnl.std() > 0:
        trades_per_year = 365.0 / max(tdf["duration_days"].mean(), 1.0)
        sharpe = (pnl.mean() / pnl.std()) * np.sqrt(trades_per_year)
    else:
        sharpe = 0.0
    return dict(
        label=label, n_trades=len(tdf),
        total_pnl=float(pnl.sum()),
        total_cost=float(tdf["cost"].sum()),
        avg_pnl=float(pnl.mean()),
        hit_rate=float((pnl > 0).mean()),
        avg_duration_days=float(tdf["duration_days"].mean()),
        sharpe=float(sharpe),
        max_dd=float(dd),
    )


print("metrics helpers defined.")

metrics helpers defined.


## ۴) Run all backtests

In [ ]:
all_trades: dict[str, list[pd.DataFrame]] = {tf: [] for tf in TIMEFRAMES}
all_metrics: dict[str, list[dict]] = {tf: [] for tf in TIMEFRAMES}
all_equities: dict[str, dict[str, pd.Series]] = {tf: {} for tf in TIMEFRAMES}

for tf in TIMEFRAMES:
    sl = shortlists[tf]
    prices = prices_by_tf[tf]
    print(f"\n=== [{tf}] running {len(sl)} backtests ===")
    for _, row in sl.iterrows():
        y_sym, x_sym = row["y"], row["x"]
        trades, equity = backtest_one_spread(
            y_sym=y_sym, x_sym=x_sym,
            alpha=row["alpha"], beta=row["beta"],
            half_life_bars=row["half_life_bars"],
            prices=prices, tf=tf,
        )
        tdf = trades_to_df(trades)
        tdf["y"] = y_sym
        tdf["x"] = x_sym
        all_trades[tf].append(tdf)

        in_df, oos_df = split_by_sample(tdf)
        m_full = metrics(tdf, tf, label="full")
        m_in   = metrics(in_df, tf, label="in_sample")
        m_oos  = metrics(oos_df, tf, label="out_of_sample")
        for m, suffix in [(m_full, ""), (m_in, "_is"), (m_oos, "_oos")]:
            m_clean = {f"{k}{suffix}": v for k, v in m.items() if k != "label"}
            m_full.update(m_clean) if suffix else None
        merged = {"y": y_sym, "x": x_sym, "beta": row["beta"], "half_life_hours": row["half_life_hours"]}
        for src, sfx in [(m_full, ""), (m_in, "_is"), (m_oos, "_oos")]:
            for k in ("n_trades", "total_pnl", "avg_pnl", "hit_rate", "sharpe", "max_dd", "avg_duration_days"):
                merged[f"{k}{sfx}"] = src[k]
        all_metrics[tf].append(merged)
        all_equities[tf][f"{y_sym}~{x_sym}"] = equity
    print(f"  done. {len(all_metrics[tf])} spreads backtested.")


=== [H1] running 59 backtests ===
  done. 59 spreads backtested.

=== [H4] running 47 backtests ===
  done. 47 spreads backtested.


## ۵) Summary metrics (sorted by OOS Sharpe)

In [ ]:
metrics_by_tf: dict[str, pd.DataFrame] = {}
for tf in TIMEFRAMES:
    m = pd.DataFrame(all_metrics[tf]).sort_values("sharpe_oos", ascending=False).reset_index(drop=True)
    metrics_by_tf[tf] = m
    print(f"\n=== [{tf}] top 15 by OUT-OF-SAMPLE Sharpe ===")
    cols = ["y", "x", "beta", "n_trades_is", "sharpe_is", "total_pnl_is",
            "n_trades_oos", "sharpe_oos", "total_pnl_oos", "hit_rate_oos",
            "avg_duration_days", "max_dd"]
    print(m[cols].head(15).to_string(index=False))

    # Aggregate
    print(f"\n  --- aggregate [{tf}] ---")
    print(f"    median Sharpe (in):    {m['sharpe_is'].median():.2f}")
    print(f"    median Sharpe (oos):   {m['sharpe_oos'].median():.2f}")
    print(f"    median PnL (in, bps):  {m['total_pnl_is'].median()*1e4:.1f}")
    print(f"    median PnL (oos, bps): {m['total_pnl_oos'].median()*1e4:.1f}")
    print(f"    median n_trades (in/oos): {m['n_trades_is'].median():.0f} / {m['n_trades_oos'].median():.0f}")
    pct_profitable_is  = (m['total_pnl_is']  > 0).mean() * 100
    pct_profitable_oos = (m['total_pnl_oos'] > 0).mean() * 100
    print(f"    % profitable spreads (in/oos):  {pct_profitable_is:.0f}% / {pct_profitable_oos:.0f}%")


=== [H1] top 15 by OUT-OF-SAMPLE Sharpe ===
     y      x    beta  n_trades_is  sharpe_is  total_pnl_is  n_trades_oos  sharpe_oos  total_pnl_oos  hit_rate_oos  avg_duration_days  max_dd
GBPUSD CADCHF -0.5141           19     1.9238        0.0970             3     26.7738         0.0344        1.0000            14.2346 -0.0218
GBPUSD NZDCHF -0.3910           19     3.1340        0.1843             2     10.5822         0.0247        1.0000            16.2212 -0.0192
NZDCAD USDCAD -0.3535           22     0.5309        0.0403             2      6.9893         0.0136        1.0000            12.6065 -0.0654
NZDCAD NZDUSD  0.4565           30     1.4981        0.0458             3      5.4413         0.0115        0.6667             9.2882 -0.0508
NZDCAD GBPAUD -0.2593           18     0.5580        0.0289             2      4.5365         0.0091        1.0000            17.7235 -0.0427
GBPNZD NZDCHF -0.7572           25     0.6742        0.0346             2      3.2080         0.0085   

## ۶) Visualize top-5 equity curves per TF

In [ ]:
for tf in TIMEFRAMES:
    m = metrics_by_tf[tf]
    top5 = m.head(5)
    if top5.empty:
        print(f"[{tf}] no spreads to plot")
        continue
    fig = go.Figure()
    for _, row in top5.iterrows():
        key = f"{row['y']}~{row['x']}"
        eq = all_equities[tf].get(key, pd.Series(dtype=float))
        if eq.empty:
            continue
        fig.add_trace(go.Scatter(x=eq.index, y=eq.values * 1e4,
                                 mode="lines", name=f"{key} (Sh_oos={row['sharpe_oos']:.2f})"))
    # Mark OOS boundary using a Scatter trace (plotly 6.x add_vline has Timestamp issues)
    if fig.data:
        ymin = min(min(tr.y) for tr in fig.data)
        ymax = max(max(tr.y) for tr in fig.data)
        fig.add_trace(go.Scatter(
            x=[OUT_OF_SAMPLE_START, OUT_OF_SAMPLE_START],
            y=[ymin, ymax],
            mode="lines", line=dict(color="red", dash="dash"),
            name="OOS start", showlegend=True,
        ))
    fig.update_layout(title=f"[{tf}] Top-5 spreads by OOS Sharpe — cumulative net PnL (bps)",
                      xaxis_title="time", yaxis_title="cumulative net PnL (bps)",
                      height=500, hovermode="x unified")
    fig.show()

## ۷) Save results

In [ ]:
for tf in TIMEFRAMES:
    m_out = STAT_DIR / f"backtest_results_{tf}.csv"
    t_out = STAT_DIR / f"backtest_trades_{tf}.csv"
    metrics_by_tf[tf].to_csv(m_out, index=False)
    pd.concat(all_trades[tf], ignore_index=True).to_csv(t_out, index=False)
    print(f"  [{tf}] metrics -> {m_out}")
    print(f"  [{tf}] trades  -> {t_out}")

print("\nnext (notebook 27): walk-forward β reestimation + portfolio construction + risk-parity sizing.")

  [H1] metrics -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\backtest_results_H1.csv
  [H1] trades  -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\backtest_trades_H1.csv
  [H4] metrics -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\backtest_results_H4.csv
  [H4] trades  -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\backtest_trades_H4.csv

next (notebook 27): walk-forward β reestimation + portfolio construction + risk-parity sizing.
